# Whisp COG Workflow

This notebook demonstrates the full workflow:
1. Export Whisp multiband image to GCS as COG
2. Create VRT from sharded COG files
3. Run exactextract on GeoJSON features

**Prerequisites:**
- Google Earth Engine authentication
- GCS credentials (for GDAL /vsigs/ access)
- Access to `whisp_bucket`

In [1]:
# Initialize Earth Engine
import ee
ee.Initialize(project='ee-andyarnellgee')  # Replace with your project

In [2]:
# Import functions
from openforis_whisp.export_cog import (
    list_cog_files_in_gcs,
    create_vrt_from_gcs,
    run_exactextract_on_cog,
)

## 1. Check Available COG Files

From the previous export (CI at 30m), we have sharded COG files in GCS.

In [ ]:
# List COG files from CI export
bucket = 'whisp_bucket'
prefix = 'whisp_cogs/CI_int16_10m'

cog_files = list_cog_files_in_gcs(bucket, prefix)
print(f"Found {len(cog_files)} COG files:")
for f in cog_files:
    print(f"  {f}")

## 2. Create VRT from Sharded COGs

Build a virtual mosaic that references all shards via `/vsigs/` paths.

In [ ]:
# Create VRT mosaic
vrt_path = create_vrt_from_gcs(bucket, prefix)
print(f"VRT created at: {vrt_path}")

In [ ]:
# Inspect VRT with rasterio
import rasterio

with rasterio.open(vrt_path) as src:
    print(f"Bands: {src.count}")
    print(f"CRS: {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Resolution: {src.res}")
    print(f"\nBand names:")
    for i, desc in enumerate(src.descriptions[:10], 1):
        print(f"  {i}: {desc}")
    if src.count > 10:
        print(f"  ... ({src.count - 10} more bands)")

## 3. Run exactextract on Test Features

Use test GeoJSON from the fixtures folder.

In [ ]:
# Path to test GeoJSON
import os
from openforis_whisp.utils import get_example_data_path

# Use example data or provide your own
geojson_path = get_example_data_path()  # Returns path to geojson_example.geojson
print(f"Using: {geojson_path}")

In [ ]:
# Run exactextract on COGs
df = run_exactextract_on_cog(
    bucket=bucket,
    prefix=prefix,
    geojson_path=geojson_path,
    ops=['sum']
)

print(f"Results shape: {df.shape}")
df.head()

## 4. Alternative: Manual VRT + exactextract

For more control, use the VRT directly with exactextract.

In [ ]:
import exactextract
import geopandas as gpd

# Load features
gdf = gpd.read_file(geojson_path)
print(f"Features: {len(gdf)}")

# Run exactextract manually
result = exactextract.exact_extract(
    rast=vrt_path,
    vec=gdf,
    ops=['sum', 'count'],
    output='pandas'
)

result.head()

## 5. Calculate Areas from Binary Sums

Since the COG contains binary (0/1) values, multiply sum by pixel area to get hectares.

In [ ]:
# At 30m resolution, each pixel = 30 * 30 = 900 m² = 0.09 ha
pixel_area_ha = (30 * 30) / 10000  # 0.09 ha

# Convert sum columns to hectares
sum_cols = [c for c in result.columns if c.endswith('_sum')]
for col in sum_cols:
    result[col.replace('_sum', '_ha')] = result[col] * pixel_area_ha

# Show area columns
ha_cols = [c for c in result.columns if c.endswith('_ha')]
result[ha_cols[:5]].head()

## Notes

**Current limitations:**
- COGs are exported as Int16 binary (0/1) for data bands, Int16 for admin_code
- Area calculation must be done client-side (sum * pixel_area)
- Large exports may shard into multiple files (VRT handles this)

**Future improvements:**
- Integrate with `local_stats.py` as alternative to GEE `getDownloadURL()`
- Add scheduled COG refresh for dynamic layers
- Pre-compute COGs for key countries